In [ ]:
import ee
import geemap
import geopandas as gpd # Import geopandas

ee.Authenticate()
ee.Initialize(project='projec_id')

In [ ]:
gdf = gpd.read_file("/content/new_study_area.shp")

aoi = geemap.geopandas_to_ee(gdf)

Map = geemap.Map()

Map.centerObject(aoi,13)

Map.addLayer(aoi,{"color":"red"},"AOI")

Map

Map(center=[10.764123272205069, 76.75266874721902], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
gdf=gpd.read_file("/content/new_study_area.shp")

AOI=geemap.geopandas_to_ee(gdf)

Map=geemap.Map()

Map.centerObject(aoi,13)

Map.addLayer(aoi,{"color":"red"},"AOI")

Map

Map(center=[10.764123272205069, 76.75266874721902], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:




Map = geemap.Map()


# ============================================================
# 3. SENTINEL-2 CLOUD MASK + SCALING
# ============================================================

def maskS2(image):

    # --------------------------------------------------------
    # QA60 cloud and cirrus mask
    # --------------------------------------------------------

    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    # --------------------------------------------------------
    # Select required Sentinel-2 bands
    #
    # B2  = Blue
    # B3  = Green
    # B4  = Red
    # B8  = NIR
    # B11 = SWIR1
    # B12 = SWIR2
    # --------------------------------------------------------

    image = image.select(
        [
            "B2",
            "B3",
            "B4",
            "B8",
            "B11",
            "B12"
        ]
    ).multiply(0.0001)

    return (
        image
        .updateMask(mask)
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )


# ============================================================
# 4. CREATE SEASONAL SPECTRAL INDICES
# ============================================================

def get_seasonal_indices(
    start_date,
    end_date,
    season_name
):

    collection = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(AOI)
        .filterDate(
            start_date,
            end_date
        )
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                60
            )
        )
        .map(maskS2)
    )

    print(
        season_name,
        "Sentinel-2 images:",
        collection.size().getInfo()
    )

    # --------------------------------------------------------
    # Median seasonal composite
    # --------------------------------------------------------

    median = collection.median()

    # --------------------------------------------------------
    # NDVI
    #
    # (NIR - RED) / (NIR + RED)
    # --------------------------------------------------------

    ndvi = (
        median
        .normalizedDifference(
            ["B8", "B4"]
        )
        .rename(
            f"{season_name}_NDVI"
        )
    )

    # --------------------------------------------------------
    # EVI
    #
    # 2.5 * (NIR - RED) /
    # (NIR + 6*RED - 7.5*BLUE + 1)
    # --------------------------------------------------------

    evi = (
        median.expression(
            "2.5 * ((NIR - RED) / " +
            "(NIR + 6 * RED - 7.5 * BLUE + 1))",
            {
                "NIR": median.select("B8"),
                "RED": median.select("B4"),
                "BLUE": median.select("B2")
            }
        )
        .rename(
            f"{season_name}_EVI"
        )
    )

    # --------------------------------------------------------
    # NDWI
    #
    # (GREEN - NIR) / (GREEN + NIR)
    #
    # Useful for water/moisture discrimination
    # --------------------------------------------------------

    ndwi = (
        median
        .normalizedDifference(
            ["B3", "B8"]
        )
        .rename(
            f"{season_name}_NDWI"
        )
    )

    # --------------------------------------------------------
    # NDMI
    #
    # (NIR - SWIR1) / (NIR + SWIR1)
    #
    # Useful for vegetation moisture
    # --------------------------------------------------------

    ndmi = (
        median
        .normalizedDifference(
            ["B8", "B11"]
        )
        .rename(
            f"{season_name}_NDMI"
        )
    )

    # --------------------------------------------------------
    # Combine all indices
    # --------------------------------------------------------

    indices = (
        ndvi
        .addBands(evi)
        .addBands(ndwi)
        .addBands(ndmi)
        .clip(AOI)
    )

    return indices


# ============================================================
# 5. DEFINE YEAR
# ============================================================

year = 2025


# ============================================================
# 6. POST-MONSOON
#
# November 2024 → February 2025
# ============================================================

post_start = f"{year - 1}-11-01"
post_end = f"{year}-03-01"

post_indices = get_seasonal_indices(
    post_start,
    post_end,
    "POST"
)


# ============================================================
# 7. PRE-MONSOON
#
# March 2025 → May 2025
# ============================================================

pre_start = f"{year}-03-01"
pre_end = f"{year}-06-01"

pre_indices = get_seasonal_indices(
    pre_start,
    pre_end,
    "PRE"
)


# ============================================================
# 8. SELECT INDIVIDUAL SEASONAL INDICES
# ============================================================

post_ndvi = post_indices.select("POST_NDVI")
post_evi = post_indices.select("POST_EVI")
post_ndwi = post_indices.select("POST_NDWI")
post_ndmi = post_indices.select("POST_NDMI")

pre_ndvi = pre_indices.select("PRE_NDVI")
pre_evi = pre_indices.select("PRE_EVI")
pre_ndwi = pre_indices.select("PRE_NDWI")
pre_ndmi = pre_indices.select("PRE_NDMI")


# ============================================================
# 9. SEASONAL CHANGE
# ============================================================

# ------------------------------------------------------------
# NDVI change

# ------------------------------------------------------------

ndvi_change = (
    pre_ndvi
    .subtract(post_ndvi)
    .rename("NDVI_CHANGE")
    .clip(AOI)
)


# ------------------------------------------------------------
# EVI change
# ------------------------------------------------------------

evi_change = (
    pre_evi
    .subtract(post_evi)
    .rename("EVI_CHANGE")
    .clip(AOI)
)


# ============================================================
# 10. SEASONAL RANGE
# ============================================================

# ------------------------------------------------------------
# NDVI range
#
# Absolute difference between the two seasons
# ------------------------------------------------------------

ndvi_range = (
    pre_ndvi
    .subtract(post_ndvi)
    .abs()
    .rename("NDVI_RANGE")
    .clip(AOI)
)


# ------------------------------------------------------------
# EVI range
# ------------------------------------------------------------

evi_range = (
    pre_evi
    .subtract(post_evi)
    .abs()
    .rename("EVI_RANGE")
    .clip(AOI)
)


# ============================================================
# 11. COMBINED FEATURE STACK
# ============================================================

seasonal_feature_stack = (
    post_indices
    .addBands(pre_indices)
    .addBands(ndvi_change)
    .addBands(evi_change)
    .addBands(ndvi_range)
    .addBands(evi_range)
)

print("\n=================================================")
print("SEASONAL FEATURE STACK")
print("=================================================")

print(
    seasonal_feature_stack.bandNames().getInfo()
)


# ============================================================
# 12. VISUALIZATION SETTINGS
# ============================================================

# ------------------------------------------------------------
# NDVI / EVI
# ------------------------------------------------------------

vegetation_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "white",
        "yellow",
        "lightgreen",
        "green",
        "darkgreen"
    ]
}


# ------------------------------------------------------------
# NDWI / NDMI
# ------------------------------------------------------------

moisture_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "brown",
        "white",
        "cyan",
        "blue",
        "darkblue"
    ]
}


# ------------------------------------------------------------
# Seasonal change
# ------------------------------------------------------------

change_vis = {
    "min": -0.4,
    "max": 0.4,
    "palette": [
        "red",
        "orange",
        "white",
        "lightgreen",
        "darkgreen"
    ]
}


# ------------------------------------------------------------
# Seasonal range
# ------------------------------------------------------------

range_vis = {
    "min": 0,
    "max": 0.4,
    "palette": [
        "white",
        "yellow",
        "orange",
        "red"
    ]
}


# ============================================================
# 13. CREATE MAP
# ============================================================

Map = geemap.Map()

Map.add_basemap("HYBRID")

Map.centerObject(
    AOI,
    10
)


# ============================================================
# 14. POST-MONSOON LAYERS
# ============================================================

Map.addLayer(
    post_ndvi,
    vegetation_vis,
    "Post-Monsoon NDVI",
    True
)

Map.addLayer(
    post_evi,
    vegetation_vis,
    "Post-Monsoon EVI",
    False
)

Map.addLayer(
    post_ndwi,
    moisture_vis,
    "Post-Monsoon NDWI",
    False
)

Map.addLayer(
    post_ndmi,
    moisture_vis,
    "Post-Monsoon NDMI",
    False
)


# ============================================================
# 15. PRE-MONSOON LAYERS
# ============================================================

Map.addLayer(
    pre_ndvi,
    vegetation_vis,
    "Pre-Monsoon NDVI",
    False
)

Map.addLayer(
    pre_evi,
    vegetation_vis,
    "Pre-Monsoon EVI",
    False
)

Map.addLayer(
    pre_ndwi,
    moisture_vis,
    "Pre-Monsoon NDWI",
    False
)

Map.addLayer(
    pre_ndmi,
    moisture_vis,
    "Pre-Monsoon NDMI",
    False
)


# ============================================================
# 16. SEASONAL CHANGE LAYERS
# ============================================================

Map.addLayer(
    ndvi_change,
    change_vis,
    "NDVI Seasonal Change",
    False
)

Map.addLayer(
    evi_change,
    change_vis,
    "EVI Seasonal Change",
    False
)


# ============================================================
# 17. SEASONAL RANGE LAYERS
# ============================================================

Map.addLayer(
    ndvi_range,
    range_vis,
    "NDVI Seasonal Range",
    False
)

Map.addLayer(
    evi_range,
    range_vis,
    "EVI Seasonal Range",
    False
)


# ============================================================
# 18. AOI
# ============================================================

Map.addLayer(
    AOI,
    {
        "color": "red"
    },
    "Palakkad AOI",
    True
)


# ============================================================
# 19. NDVI LEGEND
# ============================================================

legend_dict = {

    "0.00 – 0.25 | Very Low Vegetation":
        "white",

    "0.25 – 0.50 | Low / Moderate Vegetation":
        "yellow",

    "0.50 – 0.75 | Dense Vegetation":
        "green",

    "0.75 – 1.00 | Very Dense Vegetation / Tree Cover":
        "darkgreen"
}


Map.add_legend(
    title="NDVI / EVI Vegetation Density",
    legend_dict=legend_dict
)


# ============================================================
# 20. DISPLAY MAP
# ============================================================

Map

POST Sentinel-2 images: 39
PRE Sentinel-2 images: 29

SEASONAL FEATURE STACK
['POST_NDVI', 'POST_EVI', 'POST_NDWI', 'POST_NDMI', 'PRE_NDVI', 'PRE_EVI', 'PRE_NDWI', 'PRE_NDMI', 'NDVI_CHANGE', 'EVI_CHANGE', 'NDVI_RANGE', 'EVI_RANGE']


Map(center=[10.764123272205069, 76.75266874721902], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# =============================================================================
# Palakkad Land Cover Classification — Google Earth Engine Pipeline
# =============================================================================
#
# Task 1: Tree / Agroforestry Cover Mapping using Sentinel-2 Time Series
#
# Classes:
#   1 = DenseForest
#   2 = Agroforestry
#   3 = SeasonalCropland
#   4 = BuiltUp
#   5 = Water
#
# Workflow:
#   1. Load manually digitized reference polygons
#   2. Create polygon-level training/validation split
#   3. Build cloud-masked Sentinel-2 composites
#   4. Calculate NDVI, EVI, NDWI and NDMI
#   5. Calculate monthly NDVI/EVI temporal variability
#   6. Build final feature stack
#   7. Sample training polygons
#   8. Train Random Forest in Earth Engine
#   9. Classify AOI
#  10. Validate using held-out polygons
#  11. Calculate accuracy and area statistics
#  12. Export classification and sample tables
#
# =============================================================================


import ee
import geemap
import geopandas as gpd
import random


# =============================================================================
# 0. INITIALIZE EARTH ENGINE
# =============================================================================

ee.Initialize()


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# -------------------------------------------------------------------------
# Your manually digitized polygon file
# -------------------------------------------------------------------------
REFERENCE_SHP = "/content/labeled_poly_new_study_area.shp"

# -------------------------------------------------------------------------
# Class field
# -------------------------------------------------------------------------
CLASS_PROP = "class"

# -------------------------------------------------------------------------
# Validation fraction
#
# 25% of polygons from each class will be held out for validation.
# IMPORTANT:
# The split is done at POLYGON level, not pixel level.
# -------------------------------------------------------------------------
VALIDATION_FRACTION = 0.25

RANDOM_SEED = 42

# -------------------------------------------------------------------------
# Sentinel-2 date ranges
# -------------------------------------------------------------------------

POST_MONSOON = (
    "2023-10-01",
    "2023-12-31"
)

SUMMER = (
    "2024-03-01",
    "2024-05-31"
)

FULL_YEAR = (
    "2023-06-01",
    "2024-05-31"
)

# -------------------------------------------------------------------------
# Sentinel-2 spectral bands
# -------------------------------------------------------------------------

S2_BANDS = [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12"
]

# -------------------------------------------------------------------------
# Random Forest
# -------------------------------------------------------------------------

N_TREES = 500


CLASS_NAMES = {
    1: "DenseForest",
    2: "Agroforestry",
    3: "SeasonalCropland",
    4: "BuiltUp",
    5: "Water"
}


# =============================================================================
# 2. LOAD MANUALLY DIGITIZED POLYGONS
# =============================================================================

print("\nLoading reference polygons...")

reference_gdf = gpd.read_file(REFERENCE_SHP)

if reference_gdf.empty:
    raise ValueError("The reference shapefile contains no features.")

if CLASS_PROP not in reference_gdf.columns:
    raise ValueError(
        f"Class field '{CLASS_PROP}' was not found. "
        f"Available fields: {list(reference_gdf.columns)}"
    )

# Remove polygons without class labels
reference_gdf = reference_gdf.dropna(subset=[CLASS_PROP]).copy()

# Convert class field to integer
reference_gdf[CLASS_PROP] = reference_gdf[CLASS_PROP].astype(int)

# Keep only expected classes
reference_gdf = reference_gdf[
    reference_gdf[CLASS_PROP].isin(CLASS_NAMES.keys())
].copy()

if reference_gdf.empty:
    raise ValueError("No polygons with classes 1–5 were found.")


print(
    f"Loaded {len(reference_gdf)} manually digitized polygons."
)

print("\nReference polygon distribution:")

for cls, name in CLASS_NAMES.items():

    count = (reference_gdf[CLASS_PROP] == cls).sum()

    print(
        f"  {cls} = {name:<20} : {count} polygons"
    )


# =============================================================================
# 3. POLYGON-LEVEL TRAINING / VALIDATION SPLIT
# =============================================================================
#
# IMPORTANT:
# We split WHOLE POLYGONS.
#
# Pixels from one polygon must never appear in both training and validation,
# otherwise the accuracy can be artificially inflated due to spatial
# autocorrelation.
#
# =============================================================================

print("\nCreating polygon-level training/validation split...")

random.seed(RANDOM_SEED)

train_indices = []
valid_indices = []

for cls in sorted(CLASS_NAMES.keys()):

    class_indices = reference_gdf.index[
        reference_gdf[CLASS_PROP] == cls
    ].tolist()

    random.shuffle(class_indices)

    n_total = len(class_indices)

    # At least one validation polygon if possible
    if n_total >= 2:
        n_valid = max(
            1,
            int(round(n_total * VALIDATION_FRACTION))
        )
    else:
        n_valid = 0

    valid_class_indices = class_indices[:n_valid]
    train_class_indices = class_indices[n_valid:]

    # Safety: if a class only has one polygon,
    # keep it in training.
    if len(train_class_indices) == 0:
        train_class_indices = valid_class_indices
        valid_class_indices = []

    train_indices.extend(train_class_indices)
    valid_indices.extend(valid_class_indices)


train_gdf = reference_gdf.loc[train_indices].copy()
valid_gdf = reference_gdf.loc[valid_indices].copy()


print(
    f"\nTraining polygons   : {len(train_gdf)}"
)

print(
    f"Validation polygons : {len(valid_gdf)}"
)


print("\nTraining distribution:")

for cls, name in CLASS_NAMES.items():

    count = (
        train_gdf[CLASS_PROP] == cls
    ).sum()

    print(
        f"  {cls} = {name:<20} : {count}"
    )


print("\nValidation distribution:")

for cls, name in CLASS_NAMES.items():

    count = (
        valid_gdf[CLASS_PROP] == cls
    ).sum()

    print(
        f"  {cls} = {name:<20} : {count}"
    )


if len(valid_gdf) == 0:
    raise ValueError(
        "No validation polygons were created. "
        "You need at least two polygons in some/all classes."
    )


# =============================================================================
# 4. CONVERT TO EARTH ENGINE FEATURE COLLECTIONS
# =============================================================================

print("\nConverting polygons to Earth Engine...")

train_fc = geemap.geopandas_to_ee(train_gdf)

valid_fc = geemap.geopandas_to_ee(valid_gdf)


# =============================================================================
# 5. DEFINE AOI
# =============================================================================

# AOI is based on the extent of all manually digitized polygons.
#
# The 200 m buffer provides a small margin around the reference polygons.

aoi = (
    train_fc
    .merge(valid_fc)
    .geometry()
    .bounds()
    .buffer(200)
)


# =============================================================================
# 6. SENTINEL-2 CLOUD MASKING
# =============================================================================

def mask_s2_clouds(img):

    scl = img.select("SCL")

    # SCL classes retained:
    #
    # 4 = Vegetation
    # 5 = Bare soil
    # 6 = Water
    # 7 = Unclassified
    #
    # Removed:
    # 1 = Saturated/defective
    # 2 = Dark features
    # 3 = Cloud shadow
    # 8 = Cloud medium probability
    # 9 = Cloud high probability
    # 10 = Cirrus
    # 11 = Snow/ice

    good = scl.remap(
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0]
    )

    return (
        img
        .updateMask(good)
        .divide(10000)
        .copyProperties(
            img,
            ["system:time_start"]
        )
    )


# =============================================================================
# 7. SENTINEL-2 COLLECTION FUNCTION
# =============================================================================

def s2_collection(start_date, end_date):

    return (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(aoi)
        .filterDate(
            start_date,
            end_date
        )
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                40
            )
        )
        .map(mask_s2_clouds)
        .select(S2_BANDS)
    )


# =============================================================================
# 8. SPECTRAL INDICES
# =============================================================================

def add_indices(img):

    # NDVI
    ndvi = (
        img
        .normalizedDifference(
            ["B8", "B4"]
        )
        .rename("NDVI")
    )

    # NDWI
    ndwi = (
        img
        .normalizedDifference(
            ["B3", "B8"]
        )
        .rename("NDWI")
    )

    # NDMI
    ndmi = (
        img
        .normalizedDifference(
            ["B8", "B11"]
        )
        .rename("NDMI")
    )

    # EVI
    evi = (
        img.expression(
            "2.5 * ((NIR - RED) / "
            "(NIR + 6 * RED - 7.5 * BLUE + 1))",
            {
                "NIR": img.select("B8"),
                "RED": img.select("B4"),
                "BLUE": img.select("B2")
            }
        )
        .rename("EVI")
    )

    return img.addBands(
        [
            ndvi,
            ndwi,
            ndmi,
            evi
        ]
    )


# =============================================================================
# 9. CHECK SENTINEL-2 AVAILABILITY
# =============================================================================

print("\nChecking Sentinel-2 image availability...")

post_count = (
    s2_collection(
        *POST_MONSOON
    )
    .size()
    .getInfo()
)

summer_count = (
    s2_collection(
        *SUMMER
    )
    .size()
    .getInfo()
)

print(
    f"Post-monsoon images : {post_count}"
)

print(
    f"Summer images       : {summer_count}"
)

if post_count == 0:
    raise RuntimeError(
        "No Sentinel-2 images found for the post-monsoon period."
    )

if summer_count == 0:
    raise RuntimeError(
        "No Sentinel-2 images found for the summer period."
    )


# =============================================================================
# 10. SEASONAL COMPOSITES
# =============================================================================

print("\nCreating seasonal composites...")


post_composite = (
    s2_collection(
        *POST_MONSOON
    )
    .map(add_indices)
    .median()
)


summer_composite = (
    s2_collection(
        *SUMMER
    )
    .map(add_indices)
    .median()
)


# =============================================================================
# 11. SELECT SEASONAL FEATURES
# =============================================================================

post_bands = post_composite.select(
    [
        "B5",
        "B6",
        "B7",
        "B8A",
        "EVI",
        "NDMI",
        "NDVI",
        "NDWI"
    ],
    [
        "POST_B5",
        "POST_B6",
        "POST_B7",
        "POST_B8A",
        "POST_EVI",
        "POST_NDMI",
        "POST_NDVI",
        "POST_NDWI"
    ]
)


summer_bands = summer_composite.select(
    [
        "B5",
        "B6",
        "B7",
        "B8A",
        "EVI",
        "NDMI",
        "NDVI",
        "NDWI"
    ],
    [
        "SUMMER_B5",
        "SUMMER_B6",
        "SUMMER_B7",
        "SUMMER_B8A",
        "SUMMER_EVI",
        "SUMMER_NDMI",
        "SUMMER_NDVI",
        "SUMMER_NDWI"
    ]
)


# =============================================================================
# 12. MONTHLY PHENOLOGY / TEMPORAL VARIABILITY
# =============================================================================
#
# This section fixes the "Image.select: Band pattern 'NDVI' was applied
# to an Image with no bands" error.
#
# If a month has no Sentinel-2 imagery, a fully masked NDVI/EVI image
# is returned instead of a zero-band image.
#
# =============================================================================

print("\nBuilding monthly phenology features...")


months = ee.List.sequence(
    0,
    11
)

start = ee.Date(
    FULL_YEAR[0]
)


def monthly_indices(month_number):

    month_start = (
        start
        .advance(
            month_number,
            "month"
        )
    )

    month_end = (
        month_start
        .advance(
            1,
            "month"
        )
    )

    collection = (
        s2_collection(
            month_start,
            month_end
        )
        .map(add_indices)
    )

    # Valid fallback image containing the required bands.
    #
    # It is completely masked, so it will not contribute values
    # to temporal statistics.

    empty_image = (
        ee.Image.constant(
            [0, 0]
        )
        .rename(
            [
                "NDVI",
                "EVI"
            ]
        )
        .updateMask(
            ee.Image.constant(0)
        )
    )

    monthly_image = ee.Image(
        ee.Algorithms.If(
            collection.size().gt(0),

            collection
            .median()
            .select(
                [
                    "NDVI",
                    "EVI"
                ]
            ),

            empty_image
        )
    )

    return monthly_image


monthly_ic = (
    ee.ImageCollection
    .fromImages(
        months.map(
            monthly_indices
        )
    )
)


# =============================================================================
# 13. NDVI TEMPORAL FEATURES
# =============================================================================

ndvi_collection = (
    monthly_ic
    .select("NDVI")
)


ndvi_mean = (
    ndvi_collection
    .mean()
    .rename(
        "NDVI_MEAN"
    )
)


ndvi_range = (
    ndvi_collection
    .max()
    .subtract(
        ndvi_collection.min()
    )
    .rename(
        "NDVI_RANGE"
    )
)


ndvi_std = (
    ndvi_collection
    .reduce(
        ee.Reducer.stdDev()
    )
    .rename(
        "NDVI_STD"
    )
)


# =============================================================================
# 14. EVI TEMPORAL FEATURES
# =============================================================================

evi_collection = (
    monthly_ic
    .select("EVI")
)


evi_mean = (
    evi_collection
    .mean()
    .rename(
        "EVI_MEAN"
    )
)


evi_range = (
    evi_collection
    .max()
    .subtract(
        evi_collection.min()
    )
    .rename(
        "EVI_RANGE"
    )
)


evi_std = (
    evi_collection
    .reduce(
        ee.Reducer.stdDev()
    )
    .rename(
        "EVI_STD"
    )
)


# =============================================================================
# 15. TEMPORAL FEATURE STACK
# =============================================================================

variability_bands = ee.Image.cat(
    [
        evi_mean,
        evi_range,
        evi_std,
        ndvi_mean,
        ndvi_range,
        ndvi_std
    ]
)


# =============================================================================
# 16. FINAL FEATURE IMAGE
# =============================================================================

feature_image = (
    ee.Image.cat(
        [
            variability_bands,
            post_bands,
            summer_bands
        ]
    )
    .clip(aoi)
)


FEATURE_BANDS = feature_image.bandNames()


print("\nFinal feature bands:")

print(
    FEATURE_BANDS.getInfo()
)

print(
    "\nNumber of features:",
    FEATURE_BANDS.size().getInfo()
)


# =============================================================================
# 17. SAMPLE TRAINING POLYGONS
# =============================================================================

print("\nSampling training polygons...")


training_samples = (
    feature_image
    .sampleRegions(
        collection=train_fc,
        properties=[
            CLASS_PROP
        ],
        scale=10,
        tileScale=4,
        geometries=False
    )
)


training_count = (
    training_samples
    .size()
    .getInfo()
)


print(
    f"Training samples: {training_count:,}"
)


if training_count == 0:
    raise RuntimeError(
        "No training samples were generated. "
        "Check the polygons, AOI, CRS and Sentinel-2 coverage."
    )


# =============================================================================
# 18. TRAIN RANDOM FOREST
# =============================================================================

print("\nTraining Random Forest in Earth Engine...")


classifier = (
    ee.Classifier
    .smileRandomForest(
        numberOfTrees=N_TREES,
        minLeafPopulation=2,
        seed=RANDOM_SEED
    )
    .train(
        features=training_samples,
        classProperty=CLASS_PROP,
        inputProperties=FEATURE_BANDS
    )
)


print("Random Forest training complete.")


# =============================================================================
# 19. CLASSIFY AOI
# =============================================================================

print("\nClassifying AOI...")


classified = (
    feature_image
    .classify(
        classifier
    )
)


# =============================================================================
# 20. HELD-OUT VALIDATION
# =============================================================================

print("\nRunning held-out validation...")


validation_samples = (
    feature_image
    .sampleRegions(
        collection=valid_fc,
        properties=[
            CLASS_PROP
        ],
        scale=10,
        tileScale=4,
        geometries=False
    )
)


validation_count = (
    validation_samples
    .size()
    .getInfo()
)


print(
    f"Validation samples: {validation_count:,}"
)


if validation_count == 0:
    raise RuntimeError(
        "No validation samples were generated."
    )


validated = (
    validation_samples
    .classify(
        classifier
    )
)


# =============================================================================
# 21. ACCURACY ASSESSMENT
# =============================================================================

error_matrix = (
    validated
    .errorMatrix(
        CLASS_PROP,
        "classification"
    )
)


print("\n")
print("=" * 70)
print("ACCURACY ASSESSMENT")
print("=" * 70)


print(
    "\nConfusion Matrix:"
)

print(
    error_matrix
    .getInfo()
)


overall_accuracy = (
    error_matrix
    .accuracy()
    .getInfo()
)


kappa = (
    error_matrix
    .kappa()
    .getInfo()
)


producer_accuracy = (
    error_matrix
    .producersAccuracy()
    .getInfo()
)


consumer_accuracy = (
    error_matrix
    .consumersAccuracy()
    .getInfo()
)


print(
    f"\nOverall Accuracy : {overall_accuracy:.4f}"
)

print(
    f"Kappa            : {kappa:.4f}"
)

print(
    "\nProducer's Accuracy:"
)

print(
    producer_accuracy
)

print(
    "\nConsumer's / User's Accuracy:"
)

print(
    consumer_accuracy
)


# =============================================================================
# 22. CLASS AREA STATISTICS
# =============================================================================
#
# Required for the Task 1 output / report.
#
# Area is calculated from the classified map in km².
#
# =============================================================================

print("\nCalculating classified area...")


area_image = (
    ee.Image.pixelArea()
    .divide(1e6)
    .addBands(
        classified
    )
)


area_by_class = (
    area_image
    .reduceRegion(
        reducer=ee.Reducer.sum()
        .group(
            groupField=1,
            groupName="class"
        ),
        geometry=aoi,
        scale=10,
        maxPixels=1e9,
        tileScale=4
    )
)


area_results = (
    area_by_class
    .getInfo()
)


print("\n")
print("=" * 70)
print("CLASSIFIED AREA")
print("=" * 70)


if area_results and "groups" in area_results:

    for group in area_results["groups"]:

        class_id = int(
            group["class"]
        )

        area_km2 = group["sum"]

        class_name = CLASS_NAMES.get(
            class_id,
            f"Class {class_id}"
        )

        print(
            f"{class_id} - "
            f"{class_name:<20} "
            f"{area_km2:.2f} km²"
        )

else:

    print(
        "No area statistics were returned."
    )


# =============================================================================
# 23. EXPORT CLASSIFIED MAP
# =============================================================================

print("\nStarting classified map export...")


ee.batch.Export.image.toDrive(
    image=classified.toByte(),
    description="Palakkad_LandCover_RF",
    folder="EarthEngine_Exports",
    fileNamePrefix="palakkad_landcover_rf",
    region=aoi,
    scale=10,
    maxPixels=1e9
).start()


# =============================================================================
# 24. EXPORT TRAINING SAMPLES
# =============================================================================

print(
    "Starting training sample export..."
)


ee.batch.Export.table.toDrive(
    collection=training_samples,
    description="Palakkad_Training_Samples_Export",
    folder="EarthEngine_Exports",
    fileNamePrefix="Palakkad_RF_Training_Samples",
    fileFormat="CSV"
).start()


# =============================================================================
# 25. EXPORT VALIDATION SAMPLES
# =============================================================================

print(
    "Starting validation sample export..."
)


ee.batch.Export.table.toDrive(
    collection=validation_samples,
    description="Palakkad_Validation_Samples_Export",
    folder="EarthEngine_Exports",
    fileNamePrefix="Palakkad_RF_Validation_Samples",
    fileFormat="CSV"
).start()


# =============================================================================
# 26. FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

print(
    f"\nNumber of features : "
    f"{FEATURE_BANDS.size().getInfo()}"
)

print(
    f"Training polygons  : "
    f"{len(train_gdf)}"
)

print(
    f"Validation polygons: "
    f"{len(valid_gdf)}"
)

print(
    f"Training samples   : "
    f"{training_count:,}"
)

print(
    f"Validation samples : "
    f"{validation_count:,}"
)

print(
    f"Overall accuracy   : "
    f"{overall_accuracy:.4f}"
)

print(
    f"Kappa              : "
    f"{kappa:.4f}"
)

print(
    "\nExports have been started."
)

print(
    "Open the Earth Engine Tasks tab "
    "to monitor the exports."
)

print(
    "\nGoogle Drive folder:"
)

print(
    "EarthEngine_Exports/"
)